In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# Load dataset

df = pd.read_csv("../processed_data/cleaned_data.csv")

print(df.head())

     ID Delivery_person_ID  Delivery_person_Age  Delivery_person_Ratings  \
0  4607     INDORES13DEL02                   37                      4.9   
1  B379     BANGRES18DEL02                   34                      4.5   
2  5D6D     BANGRES19DEL01                   23                      4.4   
3  7A6A    COIMBRES13DEL02                   38                      4.7   
4  70A2     CHENRES12DEL01                   32                      4.6   

   Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
0            22.745049             75.892471                   22.765049   
1            12.913041             77.683237                   13.043041   
2            12.914264             77.678400                   12.924264   
3            11.003669             76.976494                   11.053669   
4            12.972793             80.249982                   13.012793   

   Delivery_location_longitude Type_of_order Type_of_vehicle  delivery_delay  \
0     

In [3]:
# Remove unnecessary columns

X = df.drop(
    ["delivery_delay", "ID", "Delivery_person_ID"],
    axis=1
)

# Target variable

y = df["delivery_delay"]

# Encode categorical columns

X = pd.get_dummies(X, drop_first=True)

print(X.head())

   Delivery_person_Age  Delivery_person_Ratings  Restaurant_latitude  \
0                   37                      4.9            22.745049   
1                   34                      4.5            12.913041   
2                   23                      4.4            12.914264   
3                   38                      4.7            11.003669   
4                   32                      4.6            12.972793   

   Restaurant_longitude  Delivery_location_latitude  \
0             75.892471                   22.765049   
1             77.683237                   13.043041   
2             77.678400                   12.924264   
3             76.976494                   11.053669   
4             80.249982                   13.012793   

   Delivery_location_longitude  distance  Type_of_order_Drinks   \
0                    75.912471  0.028284                  False   
1                    77.813237  0.183848                  False   
2                    77.688400  0.0

In [4]:
# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(36474, 13)
(9119, 13)


In [5]:
# Train Decision Tree model

decision_tree = DecisionTreeRegressor(
    random_state=42
)

decision_tree.fit(X_train, y_train)

print("Decision Tree training completed")

Decision Tree training completed


In [6]:
# Train Random Forest model

random_forest = RandomForestRegressor(
    random_state=42
)

random_forest.fit(X_train, y_train)

print("Random Forest training completed")

Random Forest training completed


In [7]:
# Decision Tree predictions

dt_predictions = decision_tree.predict(X_test)

# Random Forest predictions

rf_predictions = random_forest.predict(X_test)

print(rf_predictions[:5])

[30.76       27.645      34.59       34.87       28.07216667]


In [8]:
# Decision Tree metrics

dt_r2 = r2_score(y_test, dt_predictions)

# Random Forest metrics

rf_r2 = r2_score(y_test, rf_predictions)

print("Decision Tree R2 Score:", dt_r2)

print("Random Forest R2 Score:", rf_r2)

Decision Tree R2 Score: -0.2110980446246531
Random Forest R2 Score: 0.3323545210748774


In [9]:
# Hyperparameter tuning for Random Forest

params = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    params,
    cv=3
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")

print(grid_search.best_params_)

Best Parameters:
{'max_depth': 5, 'n_estimators': 100}


In [10]:
# Save best model

best_model = grid_search.best_estimator_

pickle.dump(
    best_model,
    open("../models/delivery_time_model.pkl", "wb")
)

print("Best model saved successfully")

Best model saved successfully


In [11]:
# Save training column names

training_columns = X_train.columns.tolist()

pickle.dump(
    training_columns,
    open("../models/model_columns.pkl", "wb")
)

print("Training columns saved successfully")

Training columns saved successfully
